In [1]:
import torch
import re
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [2]:
dataset = load_dataset('uclanlp/wino_bias', 'type1_anti')

data = dataset["test"]

print(dataset)
print(data[0])   # inspect structure

README.md: 0.00B [00:00, ?B/s]

type1_anti/validation-00000-of-00001.par(…):   0%|          | 0.00/31.6k [00:00<?, ?B/s]

type1_anti/test-00000-of-00001.parquet:   0%|          | 0.00/33.8k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/396 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/396 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['document_id', 'part_number', 'word_number', 'tokens', 'pos_tags', 'parse_bit', 'predicate_lemma', 'predicate_framenet_id', 'word_sense', 'speaker', 'ner_tags', 'verbal_predicates', 'coreference_clusters'],
        num_rows: 396
    })
    test: Dataset({
        features: ['document_id', 'part_number', 'word_number', 'tokens', 'pos_tags', 'parse_bit', 'predicate_lemma', 'predicate_framenet_id', 'word_sense', 'speaker', 'ner_tags', 'verbal_predicates', 'coreference_clusters'],
        num_rows: 396
    })
})
{'document_id': 'nw/test_type1/not_stereotype//0', 'part_number': '0', 'word_number': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 'tokens': ['The', 'janitor', 'reprimanded', 'the', 'accountant', 'because', 'he', 'made', 'a', 'mistake', 'filing', 'paperwork', '.'], 'pos_tags': [54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54, 54], 'parse_bit': ['-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-', '-'], 'predicate_lemma

In [3]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

model.eval()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [4]:
def mask_pronoun(tokens):

    label = None
    masked_tokens = tokens.copy()

    for i, token in enumerate(tokens):

        if token.lower() == "he":
            label = "he"
            masked_tokens[i] = "[MASK]"
            break

        if token.lower() == "she":
            label = "she"
            masked_tokens[i] = "[MASK]"
            break

    if label is None:
        return None, None

    sentence = " ".join(masked_tokens)

    return sentence, label

In [5]:
def bert_predict(sentence):

    inputs = tokenizer(sentence, return_tensors="pt")

    mask_index = torch.where(inputs["input_ids"][0] == tokenizer.mask_token_id)[0]

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    mask_logits = logits[0, mask_index, :]

    he_id = tokenizer.convert_tokens_to_ids("he")
    she_id = tokenizer.convert_tokens_to_ids("she")

    he_score = mask_logits[0][he_id]
    she_score = mask_logits[0][she_id]

    if he_score > she_score:
        return "he"
    else:
        return "she"

In [6]:
correct = 0

male_correct = 0
female_correct = 0

male_total = 0
female_total = 0

In [7]:
for sample in data:

    tokens = sample["tokens"]

    masked_sentence, label = mask_pronoun(tokens)

    if masked_sentence is None:
        continue

    pred = bert_predict(masked_sentence)

    if pred == label:
        correct += 1

    if label == "he":
        male_total += 1

        if pred == label:
            male_correct += 1

    if label == "she":
        female_total += 1

        if pred == label:
            female_correct += 1

In [8]:
accuracy = correct / (male_total + female_total)

male_accuracy = male_correct / male_total
female_accuracy = female_correct / female_total

gender_accuracy_gap = male_accuracy - female_accuracy

In [9]:
print("Total samples:", male_total + female_total)

print("Overall Accuracy:", accuracy)

print("Male Accuracy:", male_accuracy)

print("Female Accuracy:", female_accuracy)

print("Gender Accuracy Gap:", gender_accuracy_gap)

Total samples: 343
Overall Accuracy: 0.5131195335276968
Male Accuracy: 0.8068181818181818
Female Accuracy: 0.20359281437125748
Gender Accuracy Gap: 0.6032253674469243
